In [13]:
import os
import re
import sys
import pickle
import pandas as pd
import spacy
from tqdm import tqdm

In [14]:
path_annotations = "../data/annotations"
path_lexicons = "../data/lexicons"
path_predictions = "../data/predictions"

os.makedirs(path_predictions, exist_ok=True)

In [15]:
negation_cues = pd.read_csv(os.path.join(path_lexicons, "negation", "negation_ALL.csv"))
uncertainty_cues = pd.read_csv(os.path.join(path_lexicons, "uncertainty", "uncertainty_ALL.csv"))
print(f"Loaded {len(negation_cues)} negation cues and {len(uncertainty_cues)} uncertainty cues")

Loaded 59 negation cues and 79 uncertainty cues


In [ ]:
df_train = pickle.load(open(os.path.join(path_annotations, "df_train.pkl"), "rb")) # Load original labeled data
print(f"Loaded {len(df_train)} annotations")

Loaded 9319 annotations


In [ ]:
def preprocess_text(text):
    """
    Preprocess the text for rule-based analysis
    """
    if not text or pd.isna(text):
        return ""
        
    text = text.lower().strip()
    
    # Try to extract the relevant section
    result = re.search(r"motiu d'ingres(.*)destinacio a l'alta", text)
    if result:
        return result.group(1).strip()
    else:
        return text.strip()


def tokenize_text(text):
    """Split text into tokens (words, punctuation)"""
    if not text:
        return []
    # Simple tokenization
    tokens = re.findall(r'\w+|[^\w\s]', text)
    return tokens


def find_cues(text, cue_lexicon):
    """
    Find all instances of cues in the text
    
    Parameters:
        text (str): Text to search in
        cue_lexicon (DataFrame): Lexicon containing cues
        
    Returns:
        list: List of dictionaries with cue information
    """
    if not text:
        return []
    
    cues = []
    text_lower = text.lower()
    
    # Sort cues by length (longer first to avoid overlaps)
    sorted_terms = sorted(cue_lexicon["term"].tolist(), key=len, reverse=True)
    
    # Find each cue term in the text
    for term in sorted_terms:
        term_lower = term.lower()
        start_pos = 0
        
        # Find all occurrences of this term
        while start_pos < len(text_lower):
            pos = text_lower.find(term_lower, start_pos)
            if pos == -1:
                break
                
            # Found a cue
            cues.append({
                "token": term,
                "start": pos,
                "end": pos + len(term_lower)
            })
            
            # Move past this occurrence
            start_pos = pos + len(term_lower)
    
    # Sort cues by position in text
    cues = sorted(cues, key=lambda x: x["start"])
    return cues


def determine_scope(text, cue):
    """
    Determine the scope of a cue
    
    Parameters:
        text (str): Full text
        cue (dict): Cue information from find_cues
        
    Returns:
        dict: Scope information
    """
    cue_start = cue["start"]
    cue_end = cue["end"]
    
    # TODO Improve scope detection

    # Determine scope direction (forward for most, backward for "sin" ...)
    backward_cues = ["sin", "sense", "excepto", "excepte", "salvo", "tret"]
    direction = "backward" if cue["token"].lower() in backward_cues else "forward"
    
    if direction == "forward":
        # Scope starts after the cue
        scope_start = cue_end
        scope_end = len(text)
        
        # Find the next punctuation or end of text
        for i in range(scope_start, len(text)):
            if text[i] in ".,;:!?":
                scope_end = i
                break
    else:
        # Backward scope ends at the cue
        scope_end = cue_start
        scope_start = 0
        
        # Find the previous punctuation or start of text
        for i in range(scope_end-1, -1, -1):
            if i < 0 or text[i] in ".,;:!?":
                scope_start = i + 1
                break
    
    # Return scope
    return {
        "start": scope_start,
        "end": scope_end,
        "text": text[scope_start:scope_end]
    }



**Document Processing Function**

The `process_document()` function generates predictions for a single document:

For each line of text in the document:
- Preprocess the text (lowercase, extract relevant section)
- Find negation cues by matching terms from the lexicon
- For each negation cue:
   - Create a NEG prediction
   - Determine the scope affected by this negation
   - Create an NSCO prediction for the scope
- Find uncertainty cues by matching terms from the lexicon
- For each uncertainty cue:
   - Create a UNC prediction
   - Determine the scope affected by this uncertainty
   - Create a USCO prediction for the scope

This approach creates predictions based only on text pattern matching, without using existing labels

In [18]:
def process_document(doc_id, document_texts):
    """
    Process a document to detect negation and uncertainty
    
    Parameters:
        doc_id (str): Document ID
        document_texts (dict): Dictionary mapping line numbers to text
        
    Returns:
        list: List of prediction dictionaries
    """
    predictions = []
    
    for line_num, text in document_texts.items():
        if not text:
            continue
            
        processed_text = preprocess_text(text) # Preprocess text
        
        neg_cues = find_cues(processed_text, negation_cues) # Find negation cues
        
        for neg_cue in neg_cues: # Process each negation cue
            predictions.append({
                "doc_id": doc_id,
                "line_number": line_num,
                "result_id": f"neg_{len(predictions)}",
                "start": neg_cue["start"],
                "end": neg_cue["end"],
                "label": "NEG",
                "text": processed_text[neg_cue["start"]:neg_cue["end"]]
            }) # Add NEG prediction
            
            scope = determine_scope(processed_text, neg_cue) # Determine scope
            
            if scope["start"] < scope["end"]: # Add NSCO prediction if scope is non-empty
                predictions.append({
                    "doc_id": doc_id,
                    "line_number": line_num,
                    "result_id": f"nsco_{len(predictions)}",
                    "start": scope["start"],
                    "end": scope["end"],
                    "label": "NSCO",
                    "text": scope["text"]
                })
        
        # Find uncertainty cues
        unc_cues = find_cues(processed_text, uncertainty_cues)
        
        # Process each uncertainty cue
        for unc_cue in unc_cues:
            # Add UNC prediction
            predictions.append({
                "doc_id": doc_id,
                "line_number": line_num,
                "result_id": f"unc_{len(predictions)}",
                "start": unc_cue["start"],
                "end": unc_cue["end"],
                "label": "UNC",
                "text": processed_text[unc_cue["start"]:unc_cue["end"]]
            })
            
            scope = determine_scope(processed_text, unc_cue)
            
            # Add USCO prediction if scope is non-empty
            if scope["start"] < scope["end"]:
                predictions.append({
                    "doc_id": doc_id,
                    "line_number": line_num,
                    "result_id": f"usco_{len(predictions)}",
                    "start": scope["start"],
                    "end": scope["end"],
                    "label": "USCO",
                    "text": scope["text"]
                })
    
    return predictions

**Text Extraction Process**

The document text extraction code works as follows:

- Create an empty dictionary `doc_texts` to store all document texts
- For each unique document ID in our dataframe:
   - Filter the dataframe to get only rows for this document
   - Create a dictionary `texts` to store lines for this document
   - For each unique line number in this document:
     - Filter to get only annotations for this line
     - Check each annotation in this line
     - This gives us the most complete text for this line
   - Store the line texts dictionary in our main document dictionary

This extraction preserves the document and line structure while giving us just the text content to work with

In [19]:
print("Extracting document texts...")
doc_texts = {} # Extract texts for each document

for doc_id in df_train["doc_id"].unique():
    doc_df = df_train[df_train["doc_id"] == doc_id] # Filter annotations for document ID
    
    texts = {}  # Group by line_number

    for line_num in doc_df["line_number"].unique(): # Iterate over unique line numbers 
        line_df = doc_df[doc_df["line_number"] == line_num]
        line_text = ""  # Extract text from annotations
        for _, row in line_df.iterrows():
            if not pd.isna(row["text"]) and len(row["text"]) > len(line_text):
                line_text = row["text"]
        
        if line_text:
            texts[line_num] = line_text
    
    doc_texts[doc_id] = texts

Extracting document texts...


In [20]:
doc_texts["19026587"]

{'19026587_0': 'habitos toxicos.',
 '19026587_11': 'afebril,',
 '19026587_2': 'para lesiones malignas ',
 '19026587_5': 'observarse defectos de replecion.',
 '19026587_6': 'estenosis focales confirmandose la existencia de las dos estenosis de uretra anterior descritas previamente.',
 '19026587_8': 'via a nivel de uretra peneana,',
 '19026587_9': 'contraindicacion preoperatoria '}

In [21]:
print("Processing documents and generating predictions...") # Process all documents and generate predictions
all_predictions = []

for doc_id, texts in tqdm(doc_texts.items(), desc="Processing documents"):
    doc_predictions = process_document(doc_id, texts)
    # print(f"Document {doc_id}: {len(doc_predictions)} predictions")
    all_predictions.extend(doc_predictions)

# TODO - Not added document index

Processing documents and generating predictions...


Processing documents: 100%|██████████| 247/247 [00:00<00:00, 1024.06it/s]


In [22]:
df_train_pred = pd.DataFrame(all_predictions)
print(f"Generated {len(df_train_pred)} predictions")

print("Prediction distribution:")
print(df_train_pred["label"].value_counts())

# Save predictions to file
output_file = os.path.join(path_predictions, "df_train_predictions.pkl")
pickle.dump(df_train_pred, open(output_file, "wb"))
print(f"Saved predictions to {output_file}")

Generated 3813 predictions
Prediction distribution:
label
NEG     1251
NSCO     959
UNC      872
USCO     731
Name: count, dtype: int64
Saved predictions to ../data/predictions/df_train_predictions.pkl


In [23]:
df_train_pred

,doc_id,line_number,result_id,start,end,label,text
0,19026587,19026587_11,neg_0,0,7,NEG,afebril
1,19026587,19026587_6,neg_1,4,6,NEG,no
2,19026587,19026587_6,nsco_2,6,107,NSCO,sis focales confirmandose la existencia de las...
3,19026587,19026587_6,neg_3,35,37,NEG,ex
4,19026587,19026587_6,nsco_4,37,107,NSCO,istencia de las dos estenosis de uretra anteri...
...,...,...,...,...,...,...,...
3808,20339886,20339886_27,nsco_9,2,8,NSCO,udados
3809,20339886,20339886_33,neg_10,0,7,NEG,afebril
3810,20339886,20339886_35,unc_11,0,11,UNC,sospecha de
3811,20339886,20339886_35,unc_12,0,8,UNC,sospecha
